# Desarrollo Práctica G9

## Inteligencia de Negocios - Componente Práctico Semana 1

### Dominio de negocio: Ciberseguridad global e indicadores digitales por país

En este notebook se desarrolla la práctica correspondiente a la configuración de un entorno ETL utilizando Python, PostgreSQL y Docker.
El dominio seleccionado es ciberseguridad, utilizando fuentes de datos relacionadas con amenazas globales, indicadores de seguridad digital y usuarios de internet por país.

# Instalación de dependencias

In [ ]:
!pip install pandas sqlalchemy psycopg2-binary python-dotenv

## Importación de dependencias

En esta sección se importan las librerías necesarias para el desarrollo del proceso ETL.
Se utiliza pandas para la lectura y análisis de archivos CSV, python-dotenv para cargar variables de entorno, os para acceder a dichas variables, y SQLAlchemy para establecer la conexión con PostgreSQL.

In [ ]:
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

## Manejo de variables de entorno

En esta sección se cargan las credenciales de conexión a PostgreSQL desde el archivo `.env`.
Esto permite evitar escribir credenciales directamente en el código y mantener una configuración más ordenada y segura.

In [ ]:
load_dotenv()

DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")

print("Base de datos:", DB_NAME)
print("Host:", DB_HOST)
print("Puerto:", DB_PORT)
print("Usuario:", DB_USER)

## Conexión a PostgreSQL

En esta sección se crea la conexión entre Python y la base de datos PostgreSQL alojada en el contenedor Docker.
Para esto se utiliza SQLAlchemy, construyendo una cadena de conexión con las variables cargadas desde el archivo `.env`.

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

try:
    with engine.connect() as connection:
        print("Conexión exitosa a PostgreSQL")
except Exception as e:
    print("Error al conectar con PostgreSQL:")
    print(e)

## Fase de extracción de datos

En esta sección se realiza la carga de las fuentes de datos seleccionadas para el proceso ETL.
Se utilizan tres archivos CSV relacionados con el dominio de ciberseguridad: amenazas globales, indicadores de ciberseguridad por país y usuarios de internet por país.

Cada archivo se carga en un DataFrame independiente utilizando la librería pandas.

In [ ]:
# Rutas de los archivos CSV
ruta_threats = "data/Global_Cybersecurity_Threats_2015-2024.csv"
ruta_cyber_security = "data/Cyber_security.csv"
ruta_internet_users = "data/internet_users_by_country_cleaned.csv"

# Carga de los archivos CSV en DataFrames
df_threats = pd.read_csv(ruta_threats)
df_cyber_security = pd.read_csv(ruta_cyber_security)
df_internet_users = pd.read_csv(ruta_internet_users)

print("Archivos CSV cargados correctamente.")

## Validación inicial de los DataFrames

Luego de cargar los archivos CSV, se valida la cantidad de filas y columnas de cada DataFrame.
Esta revisión inicial permite confirmar que las fuentes fueron leídas correctamente antes de continuar con el análisis y la carga hacia PostgreSQL.

In [ ]:
print("Dimensiones de df_threats:", df_threats.shape)
print("Dimensiones de df_cyber_security:", df_cyber_security.shape)
print("Dimensiones de df_internet_users:", df_internet_users.shape)

In [ ]:
print("Columnas de df_threats:")
print(df_threats.columns)

print("\nColumnas de df_cyber_security:")
print(df_cyber_security.columns)

print("\nColumnas de df_internet_users:")
print(df_internet_users.columns)

## Vista preliminar de los datos

En esta sección se muestran las primeras cinco filas de cada DataFrame.
Esto permite revisar visualmente la estructura de los datos, los nombres de las columnas y algunos valores de ejemplo.

In [ ]:
df_threats.head()

In [ ]:
df_cyber_security.head()

In [ ]:
df_internet_users.head()

## Carga de datos originales hacia PostgreSQL

En esta sección se cargan los DataFrames originales hacia PostgreSQL.
Estas tablas representan la capa raw del proceso ETL, es decir, los datos tal como fueron extraídos desde los archivos CSV.

Mantener una capa raw permite conservar trazabilidad sobre las fuentes originales antes de aplicar transformaciones o limpieza de datos.

In [ ]:
# Carga de los DataFrames originales hacia PostgreSQL como tablas raw

df_threats.to_sql(
    "raw_global_cybersecurity_threats",
    engine,
    if_exists="replace",
    index=False
)

df_cyber_security.to_sql(
    "raw_cyber_security_index",
    engine,
    if_exists="replace",
    index=False
)

df_internet_users.to_sql(
    "raw_internet_users_by_country",
    engine,
    if_exists="replace",
    index=False
)

print("Tablas raw cargadas correctamente en PostgreSQL.")

## Transformación y limpieza de datos

En esta sección se generan copias transformadas de los DataFrames originales.
El objetivo es estandarizar nombres de columnas, homologar nombres de países y preparar los datos para análisis posteriores.

Se mantiene la capa raw en PostgreSQL y se crea una capa clean con datos más consistentes.

In [ ]:
# Crear copias para transformación
df_threats_clean = df_threats.copy()
df_cyber_security_clean = df_cyber_security.copy()
df_internet_users_clean = df_internet_users.copy()

In [ ]:
# Renombrar columnas del dataset de amenazas globales
df_threats_clean = df_threats_clean.rename(columns={
    "Country": "country",
    "Year": "year",
    "Attack Type": "attack_type",
    "Target Industry": "target_industry",
    "Financial Loss (in Million $)": "financial_loss_million_usd",
    "Number of Affected Users": "affected_users",
    "Attack Source": "attack_source",
    "Security Vulnerability Type": "security_vulnerability_type",
    "Defense Mechanism Used": "defense_mechanism_used",
    "Incident Resolution Time (in Hours)": "incident_resolution_time_hours"
})

# Renombrar columnas del dataset de indicadores de ciberseguridad
df_cyber_security_clean = df_cyber_security_clean.rename(columns={
    "Country": "country",
    "Region": "region",
    "CEI": "cei",
    "GCI": "gci",
    "NCSI": "ncsi",
    "DDL": "ddl"
})

# Renombrar columnas del dataset de usuarios de internet
df_internet_users_clean = df_internet_users_clean.rename(columns={
    "Country or area": "country",
    "Subregion": "subregion",
    "Region": "region",
    "Internet users": "internet_users",
    "Population_2021": "population_2021",
    "Year": "year",
    "Percentage_Internet_Users": "percentage_internet_users"
})

print("Columnas renombradas correctamente.")

In [ ]:
# Diccionario para homologar nombres de países
country_mapping = {
    "USA": "United States",
    "UK": "United Kingdom"
}

# Aplicar homologación en el dataset principal de amenazas
df_threats_clean["country"] = df_threats_clean["country"].replace(country_mapping)

print("Países homologados correctamente.")

In [ ]:
print("Valores nulos en df_threats_clean:")
print(df_threats_clean.isnull().sum())

print("\nValores nulos en df_cyber_security_clean:")
print(df_cyber_security_clean.isnull().sum())

print("\nValores nulos en df_internet_users_clean:")
print(df_internet_users_clean.isnull().sum())

In [ ]:
# Crear columna de control para identificar países con indicadores incompletos
df_cyber_security_clean["has_missing_security_index_data"] = df_cyber_security_clean[
    ["cei", "gci", "ncsi", "ddl"]
].isnull().any(axis=1)

df_cyber_security_clean.head()

## Carga de datos transformados hacia PostgreSQL

Luego de aplicar las transformaciones necesarias, se cargan los DataFrames limpios hacia PostgreSQL.
Estas tablas representan la capa clean del proceso ETL y contienen datos preparados para el análisis.

In [ ]:
# Carga de los DataFrames transformados hacia PostgreSQL como tablas clean

df_threats_clean.to_sql(
    "clean_global_cybersecurity_threats",
    engine,
    if_exists="replace",
    index=False
)

df_cyber_security_clean.to_sql(
    "clean_cyber_security_index",
    engine,
    if_exists="replace",
    index=False
)

df_internet_users_clean.to_sql(
    "clean_internet_users_by_country",
    engine,
    if_exists="replace",
    index=False
)

print("Tablas clean cargadas correctamente en PostgreSQL.")

In [ ]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

In [ ]:
tables_to_check = [
    "raw_global_cybersecurity_threats",
    "raw_cyber_security_index",
    "raw_internet_users_by_country",
    "clean_global_cybersecurity_threats",
    "clean_cyber_security_index",
    "clean_internet_users_by_country"
]

for table in tables_to_check:
    query = f"SELECT COUNT(*) AS total_registros FROM {table};"
    result = pd.read_sql(query, engine)
    print(f"{table}: {result['total_registros'][0]} registros")

## Análisis exploratorio de los DataFrames

En esta sección se realiza el análisis exploratorio de los tres DataFrames principales utilizados en la práctica.
El objetivo es comprender la estructura de cada fuente de datos, validar la existencia de valores nulos y obtener estadísticas básicas de variables numéricas relevantes.

Para cada DataFrame se realizan las siguientes operaciones:
- Mostrar las primeras cinco filas.
- Identificar columnas con valores nulos.
- Contar valores faltantes por columna.
- Calcular promedio, máximo y mínimo de una variable numérica.
- Obtener valores máximos y mínimos de dos variables agrupadas.

## Análisis del DataFrame: Global Cybersecurity Threats

Este DataFrame contiene información sobre incidentes globales de ciberseguridad ocurridos entre 2015 y 2024.
Incluye variables como país, año, tipo de ataque, industria afectada, pérdida financiera, usuarios afectados, vulnerabilidad explotada, mecanismo de defensa y tiempo de resolución del incidente.

In [ ]:
# Mostrar las primeras 5 filas del DataFrame de amenazas globales
df_threats_clean.head()

In [ ]:
# Identificar qué columnas contienen valores nulos
df_threats_clean.isnull().any()

In [ ]:
# Contar cuántos valores faltantes existen en cada columna
df_threats_clean.isnull().sum()

In [ ]:
# Calcular promedio, máximo y mínimo de la pérdida financiera
df_threats_clean["financial_loss_million_usd"].agg(["mean", "max", "min"])

In [ ]:
# Calcular pérdida financiera máxima y mínima agrupada por país
df_threats_clean.groupby("country")["financial_loss_million_usd"].agg(["max", "min"])

In [ ]:
# Calcular usuarios afectados máximos y mínimos agrupados por tipo de ataque
df_threats_clean.groupby("attack_type")["affected_users"].agg(["max", "min"])

## Análisis del DataFrame: Cyber Security Index

Este DataFrame contiene indicadores de ciberseguridad por país.
Incluye datos como región, Cybersecurity Exposure Index, Global Cybersecurity Index, National Cyber Security Index y Digital Development Level.

Este conjunto de datos permite complementar el análisis de incidentes con indicadores relacionados con exposición, madurez y desarrollo digital.

In [ ]:
# Mostrar las primeras 5 filas del DataFrame de indicadores de ciberseguridad
df_cyber_security_clean.head()

In [ ]:
# Identificar qué columnas contienen valores nulos
df_cyber_security_clean.isnull().any()

In [ ]:
# Contar cuántos valores faltantes existen en cada columna
df_cyber_security_clean.isnull().sum()

In [ ]:
# Calcular promedio, máximo y mínimo del Global Cybersecurity Index
df_cyber_security_clean["gci"].agg(["mean", "max", "min"])

In [ ]:
# Calcular GCI máximo y mínimo agrupado por región
df_cyber_security_clean.groupby("region")["gci"].agg(["max", "min"])

In [ ]:
# Calcular NCSI máximo y mínimo agrupado por región
df_cyber_security_clean.groupby("region")["ncsi"].agg(["max", "min"])

## Análisis del DataFrame: Internet Users by Country

Este DataFrame contiene información sobre usuarios de internet, población y porcentaje de penetración de internet por país.
Esta fuente permite analizar el contexto digital de cada país y relacionarlo posteriormente con los incidentes de ciberseguridad.

In [ ]:
# Mostrar las primeras 5 filas del DataFrame de usuarios de internet por país
df_internet_users_clean.head()

In [ ]:
# Identificar qué columnas contienen valores nulos
df_internet_users_clean.isnull().any()

In [ ]:
# Contar cuántos valores faltantes existen en cada columna
df_internet_users_clean.isnull().sum()

In [ ]:
# Calcular promedio, máximo y mínimo del porcentaje de usuarios de internet
df_internet_users_clean["percentage_internet_users"].agg(["mean", "max", "min"])

In [ ]:
# Calcular usuarios de internet máximos y mínimos agrupados por región
df_internet_users_clean.groupby("region")["internet_users"].agg(["max", "min"])

In [ ]:
# Calcular población máxima y mínima agrupada por subregión
df_internet_users_clean.groupby("subregion")["population_2021"].agg(["max", "min"])

## Revisión de tipos de datos

En esta sección se revisan los tipos de datos de cada DataFrame.
Esto permite verificar que las variables numéricas, categóricas y temporales hayan sido interpretadas correctamente por pandas.

In [ ]:
print("Tipos de datos - Global Cybersecurity Threats")
print(df_threats_clean.dtypes)

print("\nTipos de datos - Cyber Security Index")
print(df_cyber_security_clean.dtypes)

print("\nTipos de datos - Internet Users by Country")
print(df_internet_users_clean.dtypes)

## Estadísticas descriptivas generales

A continuación se muestran estadísticas descriptivas de las variables numéricas de cada DataFrame.
Esto permite observar medidas como promedio, desviación estándar, mínimo, máximo y cuartiles.

In [ ]:
df_threats_clean.describe()

In [ ]:
df_cyber_security_clean.describe()

In [ ]:
df_internet_users_clean.describe()

# Aplicación Profesional de la Práctica

## Guevara Paz y Miño Juan Diego
En mi contexto profesional dentro del área de tecnología, infraestructura y ciberseguridad, esta práctica tiene una aplicación muy importante porque permite relacionar el almacenamiento, integración y análisis de datos con situaciones reales que se presentan en una organización. En el área de TI se generan datos constantemente: eventos de seguridad, alertas de firewall, registros de VPN, reportes de antivirus, respaldos, accesos de usuarios, disponibilidad de servidores, consumo de recursos en la nube y métricas de servicios críticos. Muchas veces esta información se encuentra dispersa en diferentes plataformas, lo que dificulta tener una visión consolidada para tomar decisiones oportunas.

El uso de PostgreSQL, Docker y Python puede ayudar a ordenar este tipo de información. PostgreSQL permite almacenar los datos de manera estructurada y consultarlos de forma centralizada. Docker facilita levantar ambientes controlados y replicables sin depender demasiado de la configuración de una computadora específica. Python, mediante librerías como Pandas, permite cargar archivos CSV, conectarse a bases de datos, limpiar información, detectar valores faltantes y generar análisis que ayuden a entender mejor el comportamiento de los datos.

La implementación de un proceso ETL en una organización aportaría varios beneficios. Primero, permitiría extraer datos desde diferentes fuentes, como firewalls, consolas de seguridad, plataformas cloud o archivos de reportes. Luego, esos datos podrían transformarse para corregir formatos, unificar nombres de columnas, eliminar errores y preparar la información para el análisis. Finalmente, la información se cargaría en una base de datos o repositorio centralizado para generar reportes, tableros de control o indicadores de gestión.

En el caso de ciberseguridad, este tipo de análisis podría ayudar a identificar qué amenazas son más frecuentes, qué áreas tienen mayor exposición, cuánto tiempo toma resolver incidentes, qué mecanismos de defensa se usan con mayor frecuencia y qué tendencias se repiten en el tiempo. Con esta información se podrían tomar mejores decisiones sobre inversión en seguridad, priorización de controles, fortalecimiento de políticas, capacitación a usuarios y monitoreo preventivo. También sería útil para construir históricos mensuales de ataques, incidentes y mitigaciones, lo cual ayudaría a presentar información más clara a la gerencia y justificar proyectos de mejora tecnológica.

## Moreno Jeremy
Este ejercicio de EDA aplicado al sector bancario en ciberseguridad me parece extraordinariamente valioso por una razón concreta: los bancos generan datos de incidentes en tiempo real, y tener un pipeline estructurado como este permite pasar del caos informacional a decisiones accionables con rapidez.
Lo que más me impresiona de este enfoque es cómo la combinación de tres fuentes — incidentes internos (equivalente a df_threads), índices de madurez por país (como GCI o NCSI) y datos de exposición digital — replica exactamente la realidad de un banco que opera en múltiples jurisdicciones. Un banco ecuatoriano con operaciones regionales necesita entender simultáneamente sus vulnerabilidades internas y el contexto de madurez del ecosistema digital en cada mercado donde opera.
El análisis de groupby() por industria me resulta especialmente potente: en banca se podría reemplazar Target Industry por tipo de producto (banca retail, corporativa, tarjetas) y descubrir qué línea de negocio sufre mayores pérdidas o tarda más en contener un incidente.
El hallazgo de los nulos en CEI (44% faltante) también me recuerda algo crítico en entornos bancarios: los datos de ciberseguridad suelen ser incompletos porque los equipos no documentan todos los incidentes por presión operativa. Detectar esos huecos antes de modelar es precisamente lo que evita tomar decisiones sobre una base estadística falsa.

## Solis Paulo

Como desarrollador de software full stack, los conocimientos adquiridos en esta práctica pueden aplicarse directamente en la gestión, integración y análisis de datos provenientes de distintos sistemas internos y externos. En el sector bancario se trabaja con información altamente sensible y estratégica, por lo que es fundamental contar con procesos ordenados, seguros y trazables para el manejo de datos.

En este entorno se generan y gestionan diferentes tipos de datos, como transacciones financieras, información de clientes, registros de autenticación, logs de aplicaciones, eventos de seguridad, alertas de monitoreo, movimientos operativos, reportes regulatorios, indicadores de riesgo y datos provenientes de canales digitales como banca web o aplicaciones móviles. Muchos de estos datos se encuentran distribuidos en distintas bases, archivos, servicios o plataformas, lo que hace necesario integrarlos para obtener una visión más completa del negocio y de la operación tecnológica.

PostgreSQL podría utilizarse como una base de datos relacional para almacenar información estructurada, centralizar registros relevantes y facilitar consultas analíticas. Docker permitiría desplegar entornos controlados y reproducibles para bases de datos, procesos ETL o servicios de análisis, evitando diferencias entre ambientes de desarrollo, pruebas y producción. Python, mediante librerías como pandas y SQLAlchemy, permitiría extraer datos desde archivos CSV, bases de datos o APIs, transformarlos, limpiarlos, validarlos y cargarlos en una base de datos para su posterior análisis.

La implementación de un proceso ETL en una institución bancaria aportaría beneficios importantes, como la automatización de tareas repetitivas, reducción de errores manuales, mejora en la calidad de los datos, integración de múltiples fuentes y disponibilidad de información confiable para la toma de decisiones. Además, permitiría mantener trazabilidad sobre el origen de los datos y aplicar reglas de validación antes de que la información sea utilizada en reportes o sistemas analíticos.

A partir del análisis de la información obtenida, se podrían resolver problemas relacionados con detección de comportamientos anómalos, identificación de fallos recurrentes en sistemas, análisis de tiempos de respuesta, monitoreo de incidentes de seguridad, evaluación de riesgos operativos y generación de reportes para áreas de negocio o cumplimiento. También se podrían tomar decisiones sobre priorización de mejoras técnicas, optimización de procesos internos, fortalecimiento de controles de seguridad y prevención de incidentes que afecten la continuidad del servicio bancario.

En conclusión, esta práctica permite comprender cómo un proceso ETL bien diseñado puede transformar datos dispersos en información útil, confiable y accionable dentro de un entorno profesional exigente como el sector bancario.


## Evidencias de ejecución

En esta sección se presentan las capturas de pantalla que evidencian la correcta configuración del entorno, la ejecución del contenedor Docker, la conexión con PostgreSQL y la carga de datos en la base de datos.

## Evidencia 1: Estructura del proyecto

![Estructura del proyecto](capturas/semana1/01_estructura_proyecto.png)

## Evidencia 2: Ejecución de Docker Compose y contenedor activo con docker ps

![Docker Compose Up](capturas/semana1/02_docker_compose_up_docker_ps.png)

## Evidencia 3: Contenedor activo en docker desktop

![Docker PS](capturas/semana1/03_docker_desktop.png)

## Evidencia 4: Conexión exitosa a PostgreSQL

![Conexión PostgreSQL](capturas/semana1/04_conexion_postgresql.png)

## Evidencia 5: Tablas cargadas en PostgreSQL

![Tablas PostgreSQL](capturas/semana1/05_tablas_postgresql.png)
![Tablas PostgreSQL](capturas/semana1/06_tablas_postgresql.png)
![Tablas PostgreSQL](capturas/semana1/07_tablas_postgresql.png)

## Evidencia 6: Análisis exploratorio de DataFrame

![Análisis DataFrame](capturas/semana1/08_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/09_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/10_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/11_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/12_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/13_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/14_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/15_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/16_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/17_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/18_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/19_analisis_dataframe.png)
![Análisis DataFrame](capturas/semana1/20_analisis_dataframe.png)